# CASMI 2026 — analog propagation over a candidate pool

**Goal:** given tandem mass spectra of an unknown molecule, output 25 candidate
structures as SMILES, best guess first.

## Why this version exists

The previous submission scored **0.130** using spectral library search. That
approach can only return a structure that already has a reference spectrum,
and the score told us exactly how rare that is:

    P(class 1) = LB / class-1 MRR = 0.130 / 0.845 = 15.4%

So **~85% of the test set was unreachable**. This version targets it.

## The three difficulty classes

| Class | Meaning | Reachable how |
|---|---|---|
| 1 | Public reference spectra exist | library search |
| 2 | Structure known (PubChem/COCONUT), no spectra | **candidate pool + analogy** |
| 3 | Structure unknown to chemistry | neither — in no database |

## The method

1. **Candidate pool.** Subtract the adduct from the measured precursor to get
   the neutral mass, then take every structure of that mass from COCONUT plus
   the training set. ~34–70 candidates, and the answer is among them 95% of
   the time.
2. **Find an analog.** `hybrid_search` matches spectra that differ by a
   *constant mass shift* — the signature of a chemical modification. Add a
   methyl group and every fragment containing it moves by the same 14 Da. This
   finds a known relative even when the molecule itself was never measured.
3. **Rank by chemical resemblance.** Score each candidate by Tanimoto
   similarity to the analog, weighted by how well the analog matched.

The unknown is never identified directly. It is located by triangulation from
a known neighbour.

## Measured on a 196-molecule natural-product holdout

| strategy | class 1 | class 2 |
|---|---:|---:|
| identity only (previous submission) | 0.8765 | 0.0921 |
| linear blend w=4 | 0.8994 | 0.5320 |
| gated t=0.7 | 0.8986 | 0.5977 |
| **analog only (this notebook)** | 0.8955 | **0.6207** |

Ranking is analog-only because **hybrid search subsumes identity search**: a
zero-mass-shift hybrid match *is* an identity match, with self-Tanimoto 1.0.
The identity channel adds nothing on class 1 and actively harms class 2, where
the true answer has no spectrum and scores 0 while wrong candidates that happen
to be in the library score above it.

## Step 1 — install dependencies offline

**On `!pip install`:** it works while you are *editing* a notebook, because
internet can be switched on in the sidebar. But this competition scores with
**internet disabled**, so `pip` cannot reach PyPI and the install would fail
mid-run. Dependencies must therefore be shipped as files.

RDKit is pinned to **2026.3.3**, the version the host scores with. The metric
notebook states the pin "decides which submissions match; it is not a
convenience" — tautomer canonicalization changes between releases, so a
different version can produce a correct answer that fails to match.

Paths are **discovered, not hardcoded**: inputs mount at
`/kaggle/input/datasets/<user>/<name>` and `/kaggle/input/competitions/<slug>`,
both deeper than the commonly-cited `/kaggle/input/<name>`.

In [ ]:
import subprocess  # runs another program and waits for it to finish
import sys         # sys.executable is the path of the running Python
from pathlib import Path

INPUT = Path("/kaggle/input")

# Log what is actually mounted before assuming anything. If a later cell fails,
# this is the first thing to read.
print("mounted under /kaggle/input:")
for p in sorted(INPUT.glob("*")):
    print("  ", p.name)

# Locate the wheels wherever the dataset landed. rglob searches subdirectories.
wheel_dirs = sorted({p.parent for p in INPUT.rglob("*.whl")})
assert wheel_dirs, (
    "no .whl under /kaggle/input — is the casmi26-deps dataset attached, and had "
    "it finished processing before this notebook was run?"
)
WHEELS = wheel_dirs[0]
print("wheels:", WHEELS)

subprocess.run(
    [
        sys.executable,              # this interpreter, not whatever is on PATH
        "-m", "pip", "install",
        "--no-index",                # never contact PyPI
        f"--find-links={WHEELS}",    # resolve from the local directory instead
        # RDKit's only real needs are numpy and Pillow, both already present.
        # Without --no-deps the resolver would hunt for them and, with
        # --no-index, find nowhere to look.
        "--no-deps",
        "ms_entropy", "rdkit",
    ],
    check=True,  # fail here with pip's message, not later with an ImportError
)

## Step 2 — the candidate pool

Converts a measured ion back to the neutral mass of the molecule, then looks
that mass up in a sorted table of 722,737 structures (COCONUT + training).

In [ ]:
"""Candidate pool: every structure a test molecule might plausibly be.

WHY THIS EXISTS
    Phase 1 could only ever answer with a structure that had a training
    spectrum. Competition class 2 is defined as "structure is known, but no
    public spectra exist", so those molecules were unreachable — not badly
    ranked, absent. The leaderboard confirmed it: we scored 0.130, which implies
    class 1 is only ~15% of the test set.

    A candidate pool fixes the retrieval half of that. We take every structure
    from COCONUT (natural products) plus the training set, and for a query we
    keep the ones whose neutral mass matches the measured precursor. Those are
    the molecules it *could* be. Ranking them is a separate problem.

HOW A MASS BECOMES A FILTER
    The instrument measures the ion, not the molecule: an [M+H]+ ion weighs the
    molecule plus one proton. Subtract the adduct's contribution and you have
    the neutral mass, which is a property of the structure alone and can be
    compared against a database.
"""

from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd

# Monoisotopic masses: the mass of each element's MOST ABUNDANT isotope, not the
# periodic-table average. A mass spectrometer resolves individual isotopes, so
# using the average weight (e.g. Cl = 35.45) would be wrong by a third of a Dalton.
MONOISOTOPIC = {
    "C": 12.000000, "H": 1.0078250319, "N": 14.0030740052, "O": 15.9949146221,
    "S": 31.97207069, "P": 30.97376151, "F": 18.9984032, "Cl": 34.96885271,
    "Br": 78.9183376, "I": 126.904468, "Si": 27.9769265327, "Se": 79.9165196,
    "B": 11.0093055, "Na": 22.98976928, "K": 38.96370668, "Li": 7.0160040,
    "Mg": 23.9850419, "Ca": 39.9625912, "Fe": 55.9349421, "Zn": 63.9291466,
    "Cu": 62.9295989, "Mn": 54.9380471, "Co": 58.9331976, "As": 74.9215942,
    "D": 2.0141017778,
}

ELECTRON = 0.00054858

# What each adduct ADDS to the neutral monoisotopic mass to give the measured
# m/z. All are singly charged, so m/z equals mass.
#
# The electron correction matters at the accuracy we need: a proton is a
# hydrogen ATOM minus its electron (1.007825 - 0.000549 = 1.007276). Ignoring
# that is a 1.5 ppm error at m/z 350 — comparable to the tolerance itself.
_H = MONOISOTOPIC["H"]
_H2O = 2 * _H + MONOISOTOPIC["O"]
ADDUCT_MASS = {
    "[M+H]+":        _H - ELECTRON,
    "[M+NH4]+":      MONOISOTOPIC["N"] + 4 * _H - ELECTRON,
    "[M-H2O+H]+":    _H - ELECTRON - _H2O,
    "[M-2H2O+H]+":   _H - ELECTRON - 2 * _H2O,
    "[M+Na]+":       MONOISOTOPIC["Na"] - ELECTRON,
    "[M+K]+":        MONOISOTOPIC["K"] - ELECTRON,
    "[M-H]-":        -_H + ELECTRON,
    "[M-H2O-H]-":    -_H + ELECTRON - _H2O,
    "[M+CH2O2-H]-":  MONOISOTOPIC["C"] + 2 * _H + 2 * MONOISOTOPIC["O"] - _H + ELECTRON,
    "[M+Cl]-":       MONOISOTOPIC["Cl"] + ELECTRON,
}

# Matches one element symbol plus an optional count: "C16" -> ("C", "16"),
# "S" -> ("S", ""). [A-Z][a-z]? is an uppercase letter optionally followed by a
# lowercase one, which is exactly how element symbols are written.
_FORMULA_TOKEN = re.compile(r"([A-Z][a-z]?)(\d*)")


def formula_mass(formula: str) -> float | None:
    """Monoisotopic mass of a molecular formula, or None if it cannot be read.

    Returns None for formulas containing an unknown element, rather than
    silently treating it as weightless — a wrong mass would quietly exclude the
    right candidate from every future search.
    """
    if not isinstance(formula, str) or not formula:
        return None
    # Charge markers and hydrates would corrupt a neutral mass, so reject them.
    if any(c in formula for c in "+-."):
        return None

    total = 0.0
    consumed = 0
    for sym, count in _FORMULA_TOKEN.findall(formula):
        if sym not in MONOISOTOPIC:
            return None
        total += MONOISOTOPIC[sym] * (int(count) if count else 1)
        consumed += len(sym) + len(count)

    # findall skips anything it cannot match, so compare the consumed length
    # against the input to catch stray characters instead of ignoring them.
    if consumed != len(formula):
        return None
    return total


def build_pool(coconut_csv: Path, train_parquet: Path, out: Path) -> pd.DataFrame:
    """Merge COCONUT and the training structures into one mass-indexed pool."""
    # --- COCONUT: the natural-product database ---------------------------
    # Only the four columns we need; the file is 577 MB and has 40 columns.
    coco = pd.read_csv(
        coconut_csv,
        usecols=["canonical_smiles", "standard_inchi_key", "molecular_formula",
                 "exact_molecular_weight"],
    )
    coco = coco.rename(columns={"canonical_smiles": "smiles"})
    coco["inchikey14"] = coco.standard_inchi_key.str.slice(0, 14)
    coco["source"] = "coconut"

    # --- training structures ---------------------------------------------
    # Already in the competition data, and they cover class 1. Included so one
    # pool serves every class.
    tr = pd.read_parquet(
        train_parquet, columns=["inchikey14", "normalized_smiles", "molecular_formula"]
    )
    tr = tr.rename(columns={"normalized_smiles": "smiles"})
    tr = tr.drop_duplicates(subset="inchikey14")
    tr["source"] = "train"
    tr["exact_molecular_weight"] = np.nan

    pool = pd.concat(
        [coco[["inchikey14", "smiles", "molecular_formula", "exact_molecular_weight",
               "source"]], tr],
        ignore_index=True,
    )

    # Drop rows with no usable key or structure before doing expensive work.
    pool = pool[pool.inchikey14.notna() & pool.smiles.notna()]

    # Prefer training entries on collision: their SMILES are RDKit-standardised
    # the same way the competition labels are. keep="last" works because the
    # training rows were concatenated after COCONUT's.
    pool = pool.drop_duplicates(subset="inchikey14", keep="last")

    # Mass from formula, which is exact and fast. Parsing a million SMILES with
    # RDKit would take half an hour and give the same answer.
    mass = pool.molecular_formula.map(formula_mass)
    pool["exact_mass"] = mass
    pool = pool[pool.exact_mass.notna()]

    # Sorting by mass is what lets a query use binary search instead of scanning
    # the whole pool. reset_index makes positions contiguous afterwards.
    pool = pool.sort_values("exact_mass").reset_index(drop=True)

    out.parent.mkdir(parents=True, exist_ok=True)
    pool[["inchikey14", "smiles", "molecular_formula", "exact_mass", "source"]].to_parquet(
        out, index=False
    )
    return pool


class CandidatePool:
    """Mass-sorted structures, queried by precursor m/z and adduct."""

    def __init__(self, path: Path):
        self.df = pd.read_parquet(path)
        # Hot arrays pulled out of pandas once; indexing numpy is far cheaper.
        self.mass = self.df.exact_mass.to_numpy()
        self.smiles = self.df.smiles.to_numpy()
        self.key = self.df.inchikey14.to_numpy()
        assert np.all(np.diff(self.mass) >= 0), "pool must be sorted by mass"

    def __len__(self) -> int:
        return len(self.df)

    def neutral_mass(
        self, precursor_mz: float, adduct: str, offset_ppm: float = 0.0
    ) -> float | None:
        """Undo the adduct: measured ion m/z -> mass of the neutral molecule.

        `offset_ppm` removes a known systematic calibration bias. Measured from
        train.parquet's `precursor_error_ppm`, every source library reads HIGH
        against the mass implied by its own labelled structure:

            enveda-180 (timsTOF)  +0.98 ppm   <- the test instrument
            gnps       (Orbitrap) +1.55 ppm
            massbank   (QTOF)     +1.87 ppm
            riken      (QTOF)     +3.29 ppm

        This is not noise, it is calibration. Correcting it lets the search
        window be CENTRED on the truth rather than straddling it, so a narrower
        window keeps the same recall — and a narrower window means fewer
        candidates competing for 25 slots.
        """
        add = ADDUCT_MASS.get(adduct)
        if add is None:
            return None
        m = precursor_mz - add
        # The measurement is high by offset_ppm, so divide it back down.
        return m / (1.0 + offset_ppm * 1e-6)

    def query(
        self,
        precursor_mz: float,
        adduct: str,
        ppm: float = 10.0,
        offset_ppm: float = 0.0,
    ) -> np.ndarray:
        """Pool positions whose mass matches this measurement within `ppm`.

        Tolerance is in parts-per-million, not Daltons, because instrument mass
        error scales with mass: 5 ppm is 0.0017 Da at m/z 350 but 0.005 Da at
        m/z 1000.
        """
        m = self.neutral_mass(precursor_mz, adduct, offset_ppm)
        if m is None or m <= 0:
            return np.empty(0, dtype=np.int64)
        tol = m * ppm * 1e-6
        # Two binary searches on the sorted mass array bound the window.
        lo, hi = np.searchsorted(self.mass, [m - tol, m + tol])
        return np.arange(lo, hi, dtype=np.int64)


def _self_check() -> None:
    # Caffeine C8H10N4O2, monoisotopic 194.08038
    m = formula_mass("C8H10N4O2")
    assert abs(m - 194.080376) < 1e-4, m
    # Glucose C6H12O6, 180.06339
    assert abs(formula_mass("C6H12O6") - 180.063388) < 1e-4
    # Multi-letter element symbols must not be read as two elements.
    assert abs(formula_mass("CCl4") - (12.0 + 4 * 34.96885271)) < 1e-6
    # Unknown or malformed input must be rejected, never silently zero-weighted.
    assert formula_mass("C6H12Xx2") is None
    assert formula_mass("C6H12O6+") is None   # charged
    assert formula_mass("C6H12O6.H2O") is None  # hydrate
    assert formula_mass("") is None

    # An [M+H]+ ion of caffeine is measured at 195.0877.
    off = ADDUCT_MASS["[M+H]+"]
    assert abs((194.080376 + off) - 195.087652) < 1e-4, 194.080376 + off
    # [M-H]- of glucose is 179.0561.
    assert abs((180.063388 + ADDUCT_MASS["[M-H]-"]) - 179.056112) < 1e-4

    print("pool self-check passed")


if __name__ == "__main__":
    _self_check()

## Step 3 — chemical similarity

Morgan fingerprints turn a structure into a bit vector of its substructures;
Tanimoto measures overlap. This is what "chemically resembles" means
numerically.

In [ ]:
"""Analog propagation: rank candidates by resemblance to a spectral cousin.

THE PROBLEM THIS SOLVES
    A competition class-2 molecule has no reference spectrum anywhere, so
    identity search returns nothing and scores exactly zero. But chemistry is
    not random: an unknown natural product almost always has a *relative* that
    has been measured.

THE CHAIN OF REASONING
    1. Hybrid search finds library spectra that match apart from a constant mass
       shift. A constant shift is the signature of a chemical modification.
    2. Those hits identify a known molecule the unknown resembles — the analog.
    3. The pool gives every structure of the right mass.
    4. Prefer candidates that look chemically like the analog.

    The unknown is never identified directly. It is located by triangulation
    from a known neighbour.
"""

from __future__ import annotations

from functools import lru_cache

import numpy as np
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")

# Morgan fingerprints (a.k.a. ECFP4 at radius 2). Each bit records the presence
# of one circular atom neighbourhood, so two molecules sharing substructures
# share bits. radius=2 means "each atom plus everything within two bonds",
# which is the standard choice for similarity work.
_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# How sharply to favour strong analogs. score**4 means an analog matching at
# 0.9 counts ~2.5x more than one at 0.7, and a weak 0.3 match is suppressed to
# nearly nothing. Without this, a crowd of poor analogs outvotes one good one.
DEFAULT_POWER = 4.0


@lru_cache(maxsize=2_000_000)
def morgan_fp(smiles: str):
    """Fingerprint for a SMILES, or None if RDKit cannot parse it.

    Cached because candidate pools overlap heavily between queries — molecules
    of similar mass keep reappearing — and parsing dominates the cost.
    """
    mol = Chem.MolFromSmiles(smiles)
    return None if mol is None else _GEN.GetFingerprint(mol)


def rank_by_analogy(
    candidate_smiles: np.ndarray,
    analog_smiles: np.ndarray,
    analog_scores: np.ndarray,
    power: float = DEFAULT_POWER,
) -> np.ndarray:
    """Score each candidate by its weighted similarity to the best analog.

    score(candidate) = max over analogs of  analog_similarity**power
                                            * Tanimoto(candidate, analog)

    `max` rather than a sum: one convincing analog is strong evidence, whereas
    summing would reward a candidate that is vaguely similar to many unrelated
    hits. Returns one score per candidate, aligned to `candidate_smiles`.
    """
    scores = np.zeros(len(candidate_smiles), dtype=np.float32)
    if len(candidate_smiles) == 0 or len(analog_smiles) == 0:
        return scores

    # Fingerprint the candidates once, keeping track of which survived parsing.
    # BulkTanimotoSimilarity cannot accept a None in its list.
    fps, keep = [], []
    for i, smi in enumerate(candidate_smiles):
        fp = morgan_fp(smi)
        if fp is not None:
            fps.append(fp)
            keep.append(i)
    if not fps:
        return scores
    keep = np.asarray(keep)

    for a_smi, a_score in zip(analog_smiles, analog_scores):
        # A weak analog cannot lift any candidate above what a strong one
        # already gave, so skip it rather than paying for the comparison.
        weight = float(a_score) ** power
        if weight <= 0:
            continue
        a_fp = morgan_fp(a_smi)
        if a_fp is None:
            continue
        # One C++ call for all candidates instead of a Python loop.
        sims = np.asarray(
            DataStructs.BulkTanimotoSimilarity(a_fp, fps), dtype=np.float32
        )
        # Plain assignment, NOT np.maximum(..., out=scores[keep]): indexing with
        # an integer array returns a copy, so an `out=` write would land in a
        # temporary and be discarded, leaving every score at zero.
        scores[keep] = np.maximum(scores[keep], weight * sims)

    return scores


def _self_check() -> None:
    caffeine = "CN1C=NC2=C1C(=O)N(C)C(=O)N2C"
    theophylline = "Cn1c(=O)c2[nH]cnc2n(C)c1=O"   # caffeine minus one methyl
    glucose = "OCC1OC(O)C(O)C(O)C1O"               # unrelated

    assert morgan_fp("not a molecule") is None
    assert morgan_fp(caffeine) is not None

    # A caffeine-like analog must rank theophylline above glucose.
    cands = np.array([glucose, theophylline])
    s = rank_by_analogy(cands, np.array([caffeine]), np.array([1.0]))
    assert s[1] > s[0], s

    # A weaker analog must produce uniformly weaker scores.
    weak = rank_by_analogy(cands, np.array([caffeine]), np.array([0.5]))
    assert weak[1] < s[1]

    # Unparseable candidates keep their slot at score 0 rather than shifting
    # the alignment of everything after them.
    s2 = rank_by_analogy(
        np.array(["@@bad@@", theophylline]), np.array([caffeine]), np.array([1.0])
    )
    assert s2[0] == 0.0 and s2[1] > 0.0, s2

    # No analogs at all is a valid state (nothing matched); score everything 0.
    assert rank_by_analogy(cands, np.array([]), np.array([])).tolist() == [0.0, 0.0]

    print("analog self-check passed")


if __name__ == "__main__":
    _self_check()

## Step 4 — the spectral library

Builds a Flash Entropy index over the training spectra and exposes both
searches: `score_spectrum` (identity) and `analog_structures` (hybrid).

In [ ]:
"""Spectral library: index the training spectra, then rank structures for a query.

This file is embedded verbatim into the Kaggle submission notebook by
`scripts/package_kaggle.py`, so it is commented for reading, not just for
maintenance. It is the whole of the Phase 1 pipeline.

THE IDEA IN THREE SENTENCES
    A mass spectrum is a set of (fragment mass, intensity) pairs. Two spectra of
    the same molecule look alike, so we can identify an unknown by finding the
    training spectrum it most resembles and returning that molecule's structure.
    "Resembles" is measured by entropy similarity, and the search is made fast by
    an index that only compares spectra whose precursor masses already match.

WHY THIS CANNOT SOLVE THE WHOLE PROBLEM
    It can only ever return a structure that is already in the training set. The
    competition deliberately includes molecules that have no public spectra
    (class 2) or are unknown to chemistry entirely (class 3). Those score zero
    here, by construction, no matter how good the similarity function is.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from ms_entropy import FlashEntropySearch

# --------------------------------------------------------------------------
# Tuning constants
# --------------------------------------------------------------------------

# Keep at most this many peaks per library spectrum. The raw data contains
# spectra with up to 73,318 peaks, nearly all of it electronic noise. Capping
# bounds index memory at roughly 1 KB per spectrum (128 peaks x 2 values x 4
# bytes) and removes noise that would only dilute the similarity score.
MAX_PEAKS = 128

# Discard peaks shorter than 1% of the tallest peak in the same spectrum.
# Detector noise lives down there; real fragments generally do not.
NOISE_THRESHOLD = 0.01

# The REAL test set, taken from the competition's data description page.
# Deliberately NOT measured from the shipped test.parquet: that file is a
# placeholder built out of training rows and is swapped for the hidden test set
# when the notebook is re-run for scoring. Fitting these constants to the
# placeholder (245-460 m/z, 7 adducts) would silently discard most of the real
# test range.
TEST_ADDUCTS = frozenset(
    {
        # An "adduct" is what the molecule picked up to become an electrically
        # charged ion, which is the only form a mass spectrometer can weigh.
        # [M+H]+ means "the molecule plus one proton"; its measured mass is
        # therefore the molecule's mass plus ~1.007.
        "[M+H]+", "[M+NH4]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M+Na]+", "[M+K]+",
        "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+Cl]-",
    }
)

# Test molecules weigh 157-1159 Da. Adduct offsets span about -19 to +46, so the
# measured precursor m/z can land anywhere in this padded window.
TEST_MZ_RANGE = (130.0, 1220.0)

# Columns read from train.parquet. Listing them explicitly matters: the two peak
# columns are ~2.7 GB of the 3 GB file, and asking for columns we do not need
# would multiply the memory cost for nothing.
LIB_COLUMNS = [
    "inchikey14",          # 14-char hash of the 2D structure - our identity key
    "normalized_smiles",   # the structure itself, as a text string
    "adduct",              # e.g. "[M+H]+"
    "instrument_type",     # e.g. "timsTOF"
    "precursor_mz",        # measured mass of the whole ionised molecule
    "ms2_mzs",             # array of fragment masses
    "ms2_normalized_intensities",  # array of fragment heights, aligned to above
]


def _top_peaks(mzs, intensities, n: int = MAX_PEAKS) -> np.ndarray:
    """Return the n most intense peaks as an (N, 2) array sorted by mass.

    ms_entropy expects peaks shaped [[mz, intensity], [mz, intensity], ...] in
    ascending mass order, so this both truncates and reorders.
    """
    # float32 halves memory versus float64 and is far finer than the instrument's
    # actual mass accuracy, so no real precision is lost.
    mz = np.asarray(mzs, dtype=np.float32)
    it = np.asarray(intensities, dtype=np.float32)

    if mz.size > n:
        # argpartition finds the n largest intensities WITHOUT sorting the rest.
        # That is O(size) instead of O(size log size) - it matters because this
        # runs two million times. The -n index means "the last n after
        # partitioning", i.e. the n biggest.
        keep = np.argpartition(it, -n)[-n:]
        mz, it = mz[keep], it[keep]

    # argpartition returned them in arbitrary order; restore ascending mass.
    order = np.argsort(mz)

    # Glue the two 1-D arrays into one (N, 2) array: column 0 mass, column 1
    # intensity.
    return np.stack([mz[order], it[order]], axis=1)


def iter_library_rows(
    train_path: Path,
    mz_range: tuple[float, float],
    adducts: set[str],
    batch_size: int = 50_000,
):
    """Stream train.parquet in batches, yielding only the rows worth indexing.

    Streaming rather than reading the file whole is not an optimisation, it is a
    requirement: the file is 3 GB on disk and expands well beyond that in memory,
    and we still have to build a multi-gigabyte index alongside it.
    """
    # ParquetFile opens the file's metadata without reading any row data.
    pf = pq.ParquetFile(train_path)
    lo, hi = mz_range

    # Running count of rows seen so far, used to give every row a stable id.
    offset = 0

    # iter_batches yields chunks instead of the whole table, so peak memory stays
    # flat regardless of file size.
    for batch in pf.iter_batches(batch_size=batch_size, columns=LIB_COLUMNS):
        df = batch.to_pandas()

        # row_id is this row's absolute position in train.parquet. Validation
        # uses it to pull a spectrum's raw peaks back out and to hide that exact
        # measurement from the library. Computed before filtering, so the numbers
        # stay aligned to the real file.
        df["row_id"] = np.arange(offset, offset + len(df), dtype=np.int64)
        offset += len(df)

        # Three filters, combined with & (elementwise AND across the whole
        # column at once - far faster than looping over rows):
        df = df[
            df.precursor_mz.between(lo, hi)     # inside the test mass window
            & df.adduct.isin(adducts)           # an adduct the test set uses
            & df.normalized_smiles.notna()      # has a structure label at all
        ]

        # Skip empty batches so the caller never handles a zero-row frame.
        if len(df):
            yield df


def build(
    train_path: Path,
    out_dir: Path,
    mz_range: tuple[float, float],
    adducts: set[str],
) -> None:
    """Build the searchable index and save it, with metadata, to out_dir."""
    # spectra: what ms_entropy indexes. meta: everything we need to interpret a
    # result (which molecule was it?). They must stay in lockstep.
    spectra, meta = [], []

    for df in iter_library_rows(train_path, mz_range, adducts):
        # itertuples is used rather than iterrows because it is roughly 10x
        # faster - it yields lightweight namedtuples instead of building a pandas
        # Series per row.
        for row in df.itertuples(index=False):
            spectra.append(
                {
                    # "id" is our own field. ms_entropy carries unknown keys
                    # through untouched, which is how we recover the original
                    # ordering below. Its value is simply this spectrum's
                    # position in the list we are building.
                    "id": len(spectra),
                    "precursor_mz": float(row.precursor_mz),
                    "peaks": _top_peaks(row.ms2_mzs, row.ms2_normalized_intensities),
                }
            )
        # Append the whole batch's metadata at once. Row order here matches the
        # order the loop above appended spectra, which is what keeps them aligned.
        meta.append(
            df[["inchikey14", "normalized_smiles", "adduct", "instrument_type", "row_id"]]
        )

    # Stitch the per-batch frames into one table. ignore_index renumbers rows
    # 0..N-1 so positions line up with the spectra list.
    meta = pd.concat(meta, ignore_index=True)
    print(f"  collected {len(spectra):,} spectra")

    fes = FlashEntropySearch()

    # build_index does the heavy lifting: cleans every spectrum, then builds the
    # lookup structures that make searching millions of spectra fast.
    ordered = fes.build_index(
        spectra,
        max_peak_num=MAX_PEAKS,
        noise_threshold=NOISE_THRESHOLD,
        clean_spectra=True,
    )

    # ---------------------------------------------------------------------
    # THE SUBTLE PART. build_index SORTS the spectra by precursor mass and
    # returns them in the new order. It may also drop spectra that end up empty
    # after cleaning. If we left `meta` in its original order, every similarity
    # score would be attributed to the WRONG molecule - and nothing would crash.
    # So we read back each surviving spectrum's "id" (its original position) and
    # reorder meta to match exactly.
    # ---------------------------------------------------------------------
    perm = np.fromiter((s["id"] for s in ordered), dtype=np.int64, count=len(ordered))
    meta = meta.iloc[perm].reset_index(drop=True)

    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "flash").mkdir(exist_ok=True)
    fes.write(out_dir / "flash")  # ms_entropy's own binary format

    # One row per unique structure. Many spectra share a structure (a
    # well-studied molecule can have hundreds), and predictions are per
    # structure, so we need this collapsed view.
    structures = (
        meta.groupby("inchikey14", sort=True)     # sort=True fixes a stable order
        .agg(smiles=("normalized_smiles", "first"))  # any of its SMILES will do
        .reset_index()
    )

    # For each indexed spectrum, which row of `structures` does it belong to?
    # get_indexer looks up every value at once and returns integer positions,
    # which is what lets the scoring step stay pure numpy.
    meta["struct_idx"] = pd.Index(structures.inchikey14).get_indexer(meta.inchikey14)

    # A -1 would mean a spectrum whose structure is missing from the table, which
    # can only happen if the two were built from different data. Fail loudly.
    assert (meta.struct_idx >= 0).all(), "every spectrum must map to a structure"

    meta.to_parquet(out_dir / "meta.parquet", index=False)
    structures.to_parquet(out_dir / "structures.parquet", index=False)
    print(f"  indexed {len(meta):,} spectra / {len(structures):,} structures")


class Library:
    """A loaded index plus the lookup tables needed to rank structures."""

    def __init__(self, out_dir: Path):
        # Construct then read: ms_entropy wants the path at construction time and
        # loads the arrays on .read().
        self.fes = FlashEntropySearch(path_data=out_dir / "flash", low_memory=False)
        self.fes.read(out_dir / "flash")

        self.meta = pd.read_parquet(out_dir / "meta.parquet")
        self.structures = pd.read_parquet(out_dir / "structures.parquet")

        # Pull the hot columns out of pandas into plain numpy arrays once.
        # Indexing a numpy array is far cheaper than indexing a DataFrame, and
        # these are touched on every single query.
        self.struct_idx = self.meta.struct_idx.to_numpy()
        self.n_structures = len(self.structures)
        self.smiles = self.structures.smiles.to_numpy()

        # Group spectra by structure so validation can hide one structure (or one
        # measurement of it) without rebuilding the whole index.
        # Sorting by struct_idx puts every structure's spectra in one contiguous
        # run; searchsorted then records where each run starts. "stable" keeps
        # equal elements in their original relative order, so results are
        # reproducible between runs.
        self._order = np.argsort(self.struct_idx, kind="stable")
        self._starts = np.searchsorted(
            self.struct_idx[self._order], np.arange(self.n_structures + 1)
        )

        # Lookup from train.parquet row_id back to a position in this index.
        self._row_index = pd.Index(self.meta.row_id)

    def spectra_of(self, struct: int) -> np.ndarray:
        """Index positions of every spectrum belonging to one structure."""
        # Slice out this structure's contiguous run set up in __init__.
        return self._order[self._starts[struct] : self._starts[struct + 1]]

    def positions_of_rows(self, row_ids) -> np.ndarray:
        """Index positions for given train.parquet row ids; unknown ids dropped."""
        pos = self._row_index.get_indexer(pd.Index(row_ids))
        return pos[pos >= 0]  # get_indexer returns -1 for "not found"

    def score_spectrum(
        self,
        precursor_mz: float,
        peaks: np.ndarray,
        exclude: np.ndarray | None = None,
    ) -> np.ndarray:
        """Score every structure against ONE query spectrum.

        Returns an array of length n_structures. `exclude` lists index positions
        to ignore; validation uses it to hide the query's own library entry,
        which would otherwise match itself at a perfect 1.0.
        """
        # Apply the same cleaning to the query that the library received, so like
        # is compared with like. This also removes peaks at or near the precursor
        # mass: that peak is the intact molecule that failed to fragment, it is
        # present in every candidate's spectrum, and so it tells us nothing.
        cleaned = self.fes.clean_spectrum_for_search(
            precursor_mz=precursor_mz, peaks=peaks, noise_threshold=NOISE_THRESHOLD
        )

        # A spectrum can be entirely noise and clean away to nothing. Scoring it
        # would crash, so return all-zeros and let the caller carry on.
        if len(cleaned) == 0:
            return np.zeros(self.n_structures, dtype=np.float32)

        # THE ACTUAL SEARCH. "identity" means only library spectra whose
        # precursor mass matches the query's are considered - the same molecule
        # must weigh the same. Returns one similarity per indexed spectrum.
        #   ms1_tolerance: how far the whole molecule's mass may differ (0.01 Da)
        #   ms2_tolerance: how far a fragment's mass may differ (0.02 Da)
        per_spectrum = self.fes.identity_search(
            precursor_mz=precursor_mz,
            peaks=cleaned,
            ms1_tolerance_in_da=0.01,
            ms2_tolerance_in_da=0.02,
        )

        if exclude is not None and len(exclude):
            # Copy first: the array may be a view into ms_entropy's own memory,
            # and writing to it would corrupt the index for later queries.
            per_spectrum = per_spectrum.copy()
            per_spectrum[exclude] = 0.0

        # Collapse per-spectrum scores into per-structure scores.
        out = np.zeros(self.n_structures, dtype=np.float32)

        # np.maximum.at scatters with a maximum: for each spectrum, write its
        # score into its structure's slot, keeping whichever is larger. The ".at"
        # form is essential - a plain out[idx] = vals would keep only the LAST
        # write per structure rather than the best.
        # A structure is judged by its single best-matching spectrum, because
        # averaging would penalise well-studied molecules for also having noisy
        # entries in the library.
        np.maximum.at(out, self.struct_idx, per_spectrum)
        return out

    def analog_structures(
        self,
        precursor_mz: float,
        peaks: np.ndarray,
        top_n: int = 50,
        exclude: np.ndarray | None = None,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Find library structures that are chemically RELATED to the query.

        `identity_search` demands the precursor masses match, so it can only
        find the same molecule. `hybrid_search` allows a constant mass shift
        between query and library spectrum, which is exactly what a chemical
        modification looks like: add a methyl group and every fragment
        containing it moves by the same 14 Da while the rest stay put.

        That makes this the tool for competition class 2 — the molecule itself
        has no reference spectrum, but a cousin of it does.

        Returns (structure indices, scores) for the top_n best analogs.
        """
        cleaned = self.fes.clean_spectrum_for_search(
            precursor_mz=precursor_mz, peaks=peaks, noise_threshold=NOISE_THRESHOLD
        )
        if len(cleaned) == 0:
            return np.empty(0, dtype=np.int64), np.empty(0, dtype=np.float32)

        # No ms1_tolerance argument here: allowing the precursor to differ is
        # the entire point of a hybrid search.
        sims = self.fes.hybrid_search(
            precursor_mz=precursor_mz, peaks=cleaned, ms2_tolerance_in_da=0.02
        )
        if exclude is not None and len(exclude):
            sims = sims.copy()
            sims[exclude] = 0.0

        per_struct = np.zeros(self.n_structures, dtype=np.float32)
        np.maximum.at(per_struct, self.struct_idx, sims)

        n = min(top_n, self.n_structures)
        top = np.argpartition(-per_struct, n - 1)[:n]
        top = top[np.argsort(-per_struct[top])]
        return top, per_struct[top]

    def rank_molecule(
        self,
        spectra: list[tuple[float, np.ndarray]],
        k: int = 25,
        exclude: np.ndarray | None = None,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Fuse a molecule's spectra into one ranked candidate list.

        `spectra` is [(precursor_mz, peaks), ...] - all measurements of the SAME
        molecule, typically at different collision energies. The competition
        scores per molecule, not per spectrum, so they must be combined.

        Returns (structure indices, scores), best first, both of length k.
        """
        total = np.zeros(self.n_structures, dtype=np.float32)

        # Sum each spectrum's per-structure scores...
        for mz, peaks in spectra:
            total += self.score_spectrum(mz, peaks, exclude=exclude)

        # ...then divide to get the mean. Using the mean rather than the maximum
        # rewards structures that match at EVERY collision energy over ones that
        # matched strongly at a single energy by chance. max(...,1) guards
        # against a molecule that somehow arrived with no spectra.
        total /= max(len(spectra), 1)

        # Never ask for more candidates than exist.
        k = min(k, self.n_structures)

        # Find the k best without sorting all ~269,000 structures. Negating turns
        # "largest" into "smallest", which is what argpartition finds.
        top = np.argpartition(-total, k - 1)[:k]

        # argpartition guarantees the right k, but in arbitrary order. Rank is the
        # entire metric here, so sort those k properly.
        top = top[np.argsort(-total[top])]

        return top, total[top]

## Step 5 — the fingerprint model

A set transformer that reads peaks and predicts which substructures the
molecule contains. Unlike analogy it needs no relative in the library, which
is why it wins where analogy fails: on a 290-molecule natural-product
holdout, MRR 0.4246 -> 0.6307 on molecules whose nearest spectral cousin is
chemically distant.

Trained locally on an RTX 4080: 800k spectra, 224,744 structures, 20 epochs,
5.5M parameters. The weights ship in the deps dataset.

In [ ]:
"""Spectrum -> molecular fingerprint: a set transformer over peaks.

WHY THIS SHAPE OF MODEL
    A spectrum is an unordered SET of (m/z, intensity) pairs, not a sequence and
    not an image. Transformer self-attention is permutation-equivariant, so it
    fits naturally: no positional encoding, peaks attend to each other, and the
    order they arrive in cannot matter.

WHY PREDICT A FINGERPRINT RATHER THAN A STRUCTURE
    Generating a SMILES string is an open-ended search over a combinatorial
    space. Predicting a fingerprint is a fixed-size multi-label classification
    with millions of labelled examples. The fingerprint is then used to RANK
    candidates that already exist, so the model never has to invent anything.

WHY IT SHOULD BEAT ANALOG PROPAGATION WHERE WE ARE WEAK
    Analogy needs a spectral cousin to exist. A fingerprint model learns the
    general mapping from fragmentation patterns to substructures, so it scores a
    molecule with no relative in the library at all — which is exactly the
    distant-analog bucket where we measured MRR 0.396.
"""

from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn


class SinEmb(nn.Module):
    """Sinusoidal embedding of a continuous value, log-spaced.

    A raw m/z of 372.14 is a poor network input: one number spanning three
    orders of magnitude. Projecting it onto sines and cosines of many
    wavelengths gives a smooth, high-dimensional code where nearby masses are
    nearby vectors — the same trick as positional encoding, applied to a
    physical quantity rather than an index.

    Wavelengths are log-spaced from 10^-2 to 10^3.2 Da so the code resolves both
    a 0.01 Da isotope gap and a 1000 Da precursor.
    """

    def __init__(self, dim: int, lo: float = -2.0, hi: float = 3.2):
        super().__init__()
        n = dim // 2
        wav = torch.pow(10.0, torch.linspace(lo, hi, n))
        # register_buffer: saved with the model and moved to GPU with .to(),
        # but not a trainable parameter.
        self.register_buffer("inv", (2 * math.pi) / wav)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        a = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(a), torch.cos(a)], dim=-1)


class Block(nn.Module):
    """Pre-norm transformer block with masked self-attention."""

    def __init__(self, d: int, heads: int, drop: float):
        super().__init__()
        self.h = heads
        self.n1 = nn.LayerNorm(d)
        self.n2 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.o = nn.Linear(d, d)
        self.ff = nn.Sequential(
            nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d)
        )
        self.drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor, pad: torch.Tensor) -> torch.Tensor:
        B, N, D = x.shape
        y = self.n1(x)
        # One projection for queries, keys and values, then split.
        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        # pad is True for padding slots; the mask must be True where attention
        # IS allowed, hence the inversion.
        mask = (~pad)[:, None, None, :]
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))
        return x + self.drop(self.ff(self.n2(x)))


class FPNet(nn.Module):
    """Predicts fingerprint-bit logits from a spectrum plus its metadata."""

    def __init__(self, n_bits: int, n_adducts: int, n_instruments: int = 4,
                 d: int = 256, layers: int = 4, heads: int = 8, drop: float = 0.1):
        super().__init__()
        self.cfg = dict(n_bits=n_bits, n_adducts=n_adducts,
                        n_instruments=n_instruments, d=d, layers=layers,
                        heads=heads, drop=drop)

        self.mz_emb = SinEmb(d)
        # Neutral loss = precursor - fragment. The SAME chemical group falling
        # off two different molecules gives different fragment masses but the
        # same neutral loss, so this view is translation-invariant and exposes
        # structure the raw m/z hides.
        self.nl_emb = SinEmb(d)
        self.peak = nn.Linear(2 * d + 1, d)   # +1 for intensity

        self.prec_emb = SinEmb(d)
        self.adduct = nn.Embedding(n_adducts, d)
        self.instrument = nn.Embedding(n_instruments, d)
        self.glob = nn.Linear(d + 1, d)       # +1 for collision energy

        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        # Mean and max pooling concatenated: mean captures the overall pattern,
        # max captures whether any single diagnostic peak is present.
        self.head = nn.Sequential(
            nn.Linear(2 * d, 1024), nn.GELU(), nn.Dropout(drop),
            nn.Linear(1024, n_bits),
        )

    def forward(self, mz, inten, pad, prec, add_id, ins_id, ce):
        # Neutral losses, floored at 0 — a fragment heavier than its precursor
        # is noise, and a negative loss has no physical meaning.
        nl = (prec[:, None] - mz).clamp(min=0)
        p = self.peak(torch.cat([self.mz_emb(mz), self.nl_emb(nl),
                                 inten.unsqueeze(-1)], dim=-1))

        # One global token carrying everything that is true of the whole
        # measurement rather than of any single peak.
        g = self.glob(torch.cat([self.prec_emb(prec),
                                 (ce / 100.0).unsqueeze(-1)], dim=-1))
        g = g + self.adduct(add_id) + self.instrument(ins_id)

        x = torch.cat([g.unsqueeze(1), p], dim=1)
        # The global token is never padding, so prepend a False column.
        pad = torch.cat([torch.zeros(len(pad), 1, dtype=torch.bool,
                                     device=pad.device), pad], dim=1)
        for b in self.blocks:
            x = b(x, pad)
        x = self.norm(x)

        # Exclude padding from pooling: -inf before max, and a masked mean.
        m = (~pad).unsqueeze(-1).float()
        mean = (x * m).sum(1) / m.sum(1).clamp(min=1)
        mx = x.masked_fill(pad.unsqueeze(-1), float("-inf")).max(1).values
        return self.head(torch.cat([mean, mx], dim=-1))


def score_candidates(logits: np.ndarray, cand_fp: np.ndarray) -> np.ndarray:
    """Rank candidates by the log-likelihood of their fingerprint.

    `logits` is the model's output for one spectrum, `cand_fp` is (n_cand,
    n_bits) of 0/1. The score is

        sum over bits of [ fp*log(p) + (1-fp)*log(1-p) ]

    i.e. how probable this exact bit pattern is under the model. A plain dot
    product would only reward bits the candidate HAS, so a molecule with many
    bits set would win by having more chances to agree; this also penalises
    bits the model expects and the candidate lacks.
    """
    p = 1.0 / (1.0 + np.exp(-logits.astype(np.float64)))
    eps = 1e-6
    lp, lq = np.log(p + eps), np.log1p(-p + eps)
    return cand_fp @ lp + (1 - cand_fp) @ lq


def select_peaks(mzs, intensities, max_peaks: int) -> tuple[np.ndarray, np.ndarray]:
    """The most intense `max_peaks` peaks, returned in ascending m/z order.

    THE SINGLE SOURCE OF TRUTH for peak selection. Training and inference used
    to implement this separately, and they disagreed: training took the top N by
    intensity, inference took `peaks[:N]` — which, because arrays are stored
    sorted by m/z, kept the LOWEST-MASS peaks and discarded every informative
    high-mass fragment. 69.8% of test spectra exceed the cap, so most of the
    test set was scored on the wrong half of its own spectrum.

    Nothing crashed and nothing looked wrong. Two implementations of one concept
    will drift; there is now one.
    """
    mz = np.asarray(mzs, dtype=np.float32)
    it = np.asarray(intensities, dtype=np.float32)
    if mz.size > max_peaks:
        keep = np.argpartition(it, -max_peaks)[-max_peaks:]
        mz, it = mz[keep], it[keep]
    order = np.argsort(mz)
    return mz[order], it[order]


def candidate_bits(smiles_list, bit_sel: np.ndarray) -> np.ndarray:
    """(n_cand, n_bits) 0/1 matrix on the SAME bit selection the model used.

    Lives here, beside the model, so inference and validation cannot drift onto
    different bit definitions — every score would then be computed against the
    wrong bits and the channel would look broken for an unrelated reason.
    """
    from rdkit import DataStructs


    out = np.zeros((len(smiles_list), len(bit_sel)), dtype=np.float64)
    for i, smi in enumerate(smiles_list):
        fp = morgan_fp(smi)
        if fp is None:
            continue
        arr = np.zeros(2048, dtype=np.uint8)
        DataStructs.ConvertToNumpyArray(fp, arr)
        out[i] = arr[bit_sel]
    return out


def predict_logits(model, spectra, adduct, adducts, instrument_id_, device="cpu",
                   max_peaks: int = 128, collision_energies=None) -> np.ndarray:
    """Mean fingerprint logits over all spectra of one molecule.

    Pooling happens in LOGIT space: these are log-odds, and averaging log-odds
    is the natural way to combine independent evidence about the same bit.

    `collision_energies` must be the REAL per-spectrum values. Training saw a
    median of 32 eV with only 14.5% zeros, so passing 0 — as this function
    silently did until the Phase 3 submission scored 0.196 against Phase 2's
    0.240 — asks the model about an energy it effectively never saw.
    """
    if collision_energies is None:
        # Not a safe default; the median of the training distribution is far
        # less wrong than 0, but callers should pass the real values.
        collision_energies = [32.0] * len(spectra)

    outs = []
    for (mz_val, peaks), ce_val in zip(spectra, collision_energies):
        sel_mz, sel_it = select_peaks(peaks[:, 0], peaks[:, 1], max_peaks)
        n = len(sel_mz)
        mz = np.zeros((1, max_peaks), np.float32)
        it = np.zeros((1, max_peaks), np.float32)
        mz[0, :n], it[0, :n] = sel_mz, sel_it
        pad = np.ones((1, max_peaks), bool)
        pad[0, :n] = False
        with torch.no_grad():
            z = model(
                torch.as_tensor(mz, device=device),
                torch.as_tensor(it, device=device),
                torch.as_tensor(pad, device=device),
                torch.as_tensor([mz_val], dtype=torch.float32, device=device),
                torch.as_tensor([adducts.index(adduct) if adduct in adducts else 0],
                                device=device),
                torch.as_tensor([instrument_id_], device=device),
                torch.as_tensor([float(ce_val)], dtype=torch.float32, device=device),
            )
        outs.append(z[0].float().cpu().numpy())
    return np.mean(outs, axis=0)


def save(model: FPNet, path, **extra) -> None:
    torch.save({"cfg": model.cfg, "state": model.state_dict(), **extra}, path)


def load(path, device="cpu") -> tuple[FPNet, dict]:
    ck = torch.load(path, map_location=device, weights_only=False)
    model = FPNet(**ck["cfg"]).to(device)
    model.load_state_dict(ck["state"])
    model.eval()
    return model, ck


def _self_check() -> None:
    torch.manual_seed(0)
    B, N, BITS = 3, 16, 64
    # eval(), not train(): dropout is stochastic, so the invariance checks below
    # would compare two different random masks and fail for the wrong reason.
    net = FPNet(n_bits=BITS, n_adducts=10, d=64, layers=2, heads=4).eval()

    mz = torch.rand(B, N) * 400
    it = torch.rand(B, N)
    pad = torch.zeros(B, N, dtype=torch.bool)
    pad[0, 10:] = True          # first spectrum has only 10 real peaks
    prec = torch.rand(B) * 400 + 100
    with torch.no_grad():
        out = net(mz, it, pad, prec, torch.zeros(B, dtype=torch.long),
              torch.zeros(B, dtype=torch.long), torch.full((B,), 30.0))
    assert out.shape == (B, BITS), out.shape
    assert torch.isfinite(out).all(), "non-finite logits"

    # Padding must not change the answer: the padded slots of spectrum 0 hold
    # arbitrary values, and masking should make them irrelevant.
    mz2, it2 = mz.clone(), it.clone()
    mz2[0, 10:] = 999.0
    it2[0, 10:] = 5.0
    out2 = net(mz2, it2, pad, prec, torch.zeros(B, dtype=torch.long),
               torch.zeros(B, dtype=torch.long), torch.full((B,), 30.0))
    assert torch.allclose(out[0], out2[0], atol=1e-5), "padding leaked into output"

    # Permuting real peaks must not change the answer either — a spectrum is a
    # set, and the architecture is only correct if it is permutation-invariant.
    perm = torch.randperm(N)
    out3 = net(mz[1:2, perm], it[1:2, perm], pad[1:2, perm], prec[1:2],
               torch.zeros(1, dtype=torch.long), torch.zeros(1, dtype=torch.long),
               torch.full((1,), 30.0))
    assert torch.allclose(out[1], out3[0], atol=1e-5), "not permutation invariant"

    # score_candidates must prefer the candidate matching the model's beliefs.
    logits = np.array([5.0, -5.0, 5.0, -5.0], dtype=np.float32)
    cands = np.array([[1, 0, 1, 0], [0, 1, 0, 1]], dtype=np.float64)
    s = score_candidates(logits, cands)
    assert s[0] > s[1], s

    print("fpnet self-check passed")


if __name__ == "__main__":
    _self_check()

## Step 6 — the ranker

Ties everything together: pool for candidates, the model for substructure
evidence, hybrid search for analogy.

The blend is `fpnet + 0.25*analog`, both z-scored. Swept on the hard bucket:

| strategy | all | hard |
|---|---:|---:|
| analog alone | 0.6427 | 0.4246 |
| fpnet alone | 0.6946 | 0.6307 |
| **fpnet + 0.25*analog** | **0.7424** | **0.6567** |
| fpnet + 1.0*analog | 0.7105 | 0.5489 |

An equal weight drowns the model exactly where the model is the only thing
that works.

In [ ]:
"""Combine identity search (class 1) and analog propagation (class 2).

The two channels answer different questions and are strong in different places:

  identity  "is this exact molecule in the spectral library?"  -> class 1
  analog    "does it resemble something in the library?"       -> class 2

Both produce a score per candidate, but not on the same footing. A correct
identity match sits near 1.0 and everything else near 0, so the channel is
almost binary. Analog scores are soft and rarely exceed ~0.5, because they are
`hybrid_similarity**4 * Tanimoto` and both factors are below 1.

Adding them with a weight below 1 keeps that asymmetry honest: a confident
identity match always wins, and analog ordering decides everything identity
leaves at zero — which is ~85% of the test set.
"""

from __future__ import annotations

import numpy as np


# How much the analog channel counts once the fingerprint model is in play.
#
# 0.5, chosen on the timsTOF holdout — the TEST INSTRUMENT. The previous value
# of 0.25 came from a natural-product holdout on the wrong instrument, and it
# is the one weight that loses to analog alone on timsTOF (0.7510 vs 0.7817).
# That mistake is what took the leaderboard from 0.240 to 0.196.
#
#   timsTOF, model v2 (256 peaks)      all      hard
#   analog alone                     0.7817    0.3006
#   fpnet alone                      0.7905    0.5304
#   fpnet + 0.25*analog              0.8289    0.5261
#   fpnet + 0.50*analog              0.8287    0.5520   <- chosen
#   fpnet + 1.00*analog              0.8162    0.4120
#
# 0.25 and 0.5 tie on the overall column; 0.5 wins the hard column, which is
# the regime the real test set is mostly in.
FPNET_ANALOG_WEIGHT = 0.5


def _z(x: np.ndarray) -> np.ndarray:
    """Z-score, so two differently-scaled channels can be added."""
    s = x.std()
    return (x - x.mean()) / s if s > 1e-12 else np.zeros_like(x)

# Chosen so a confident identity match (~1.0) cannot be displaced by the best
# realistic analog score (~0.5 * 0.5 = 0.25). Validated by checking that class-1
# MRR does not move when the analog channel is switched on.
ANALOG_WEIGHT = 0.5

# Swept on a 200-molecule timsTOF holdout (scripts/validate_class2.py):
#   5 ppm, no offset : 64 candidates, 100.0% recall, MRR 0.7121
#   4 ppm, +0.98 ppm : 57 candidates, 100.0% recall, MRR 0.7163  <- chosen
#   3 ppm, +0.98 ppm : 52 candidates,  99.5% recall, MRR 0.7165
# 4 ppm keeps full recall while cutting candidates 11%; 3 ppm trades 0.5% of
# recall for no measurable gain, so it is not worth the risk on unseen data.
TOP_ANALOGS = 100   # 100 beat 50 by +0.012 on the calibrated metric
PPM = 4.0
# timsTOF reads +0.98 ppm high against the mass implied by the labelled
# structure. Correcting it centres the search window instead of straddling.
MZ_OFFSET_PPM = 0.98
K = 25


class CombinedRanker:
    """Ranks pool candidates using both spectral channels."""

    def __init__(self, lib: Library, pool: CandidatePool,
                 analog_weight: float = ANALOG_WEIGHT,
                 offset_ppm: float = MZ_OFFSET_PPM,
                 fpnet=None, fpnet_ck=None, device: str = "cpu"):
        self.lib = lib
        self.pool = pool
        self.analog_weight = analog_weight
        # Optional so the pipeline still runs without a trained model.
        self.fpnet = fpnet
        self.device = device
        if fpnet is not None:
            self.bit_sel = np.asarray(fpnet_ck["bit_sel"])
            self.adducts = list(fpnet_ck["adduct_list"])
            # The cap the model was TRAINED at. Guessing 128 for a 256-peak
            # model silently truncates every rich spectrum, and 69.8% of test
            # spectra exceed 128 peaks.
            self.max_peaks = int(fpnet_ck.get("max_peaks", 128))
        # Held on the instance rather than threaded through every call: it is a
        # property of the instrument, constant for a whole run.
        self.offset_ppm = offset_ppm

        # The two channels live in different index spaces: identity scores are
        # per LIBRARY structure, candidates are per POOL row. InChIKey14 is the
        # only thing they share, so build the bridge once rather than per query.
        self._lib_row_of_key = {
            k: i for i, k in enumerate(lib.structures.inchikey14.to_numpy())
        }

    def score_components(
        self,
        spectra: list[tuple[float, np.ndarray]],
        adduct: str,
        ppm: float = PPM,
        exclude: np.ndarray | None = None,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Return (pool positions, identity scores, analog scores).

        Kept separate from `rank` so validation can sweep the blend weight
        without paying for the spectral searches again — the analog channel is
        ~40x slower than identity and dominates the cost.
        """
        mz = spectra[0][0]
        cand = (
            self.pool.query(mz, adduct, ppm=ppm, offset_ppm=self.offset_ppm)
            if adduct in ADDUCT_MASS
            else np.empty(0, np.int64)
        )
        empty = np.empty(0, dtype=np.float32)
        if len(cand) == 0:
            return cand, empty, empty

        # --- channel 1: identity ------------------------------------------
        # Mean across the molecule's spectra, matching Phase 1.
        ident_struct = np.zeros(self.lib.n_structures, dtype=np.float32)
        for q_mz, peaks in spectra:
            ident_struct += self.lib.score_spectrum(q_mz, peaks, exclude=exclude)
        ident_struct /= len(spectra)

        # Project library-structure scores onto the candidate list. A candidate
        # absent from the library simply scores 0 here, which is correct: no
        # reference spectrum means no identity evidence.
        ident = np.zeros(len(cand), dtype=np.float32)
        for i, key in enumerate(self.pool.key[cand]):
            row = self._lib_row_of_key.get(key)
            if row is not None:
                ident[i] = ident_struct[row]

        # --- channel 2: analog --------------------------------------------
        best = np.zeros(self.lib.n_structures, dtype=np.float32)
        for q_mz, peaks in spectra:
            idx, sc = self.lib.analog_structures(
                q_mz, peaks, top_n=TOP_ANALOGS, exclude=exclude
            )
            np.maximum.at(best, idx, sc)
        n = min(TOP_ANALOGS, self.lib.n_structures)
        a_idx = np.argpartition(-best, n - 1)[:n]
        a_idx = a_idx[np.argsort(-best[a_idx])]

        analog = rank_by_analogy(
            self.pool.smiles[cand], self.lib.smiles[a_idx], best[a_idx]
        )
        return cand, ident, analog

    def rank(
        self,
        spectra: list[tuple[float, np.ndarray]],
        adduct: str,
        k: int = K,
        ppm: float = PPM,
        exclude: np.ndarray | None = None,
        collision_energies: list[float] | None = None,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Return (pool positions, scores) for the k best candidates.

        With a model loaded, ranking is `fpnet + 0.5*analog` (see
        FPNET_ANALOG_WEIGHT). Without one it falls back to analog-only, which
        is the configuration that scored 0.240.

        `collision_energies` must be the real per-spectrum values; the model was
        trained with them and defaults to the training median otherwise.

        The analog-only choice was itself measured, on a 196-molecule
        natural-product holdout (scripts/validate_combined.py):

            strategy        class 1   class 2   projected LB
            identity only    0.8765    0.0921         0.1746
            linear w=4       0.8994    0.5320         0.3673
            gated t=0.7      0.8986    0.5977         0.3954
            analog only      0.8955    0.6207         0.4048   <-

        Hybrid search subsumes identity search: a zero-mass-shift hybrid match
        IS an identity match, and a molecule's Tanimoto to itself is 1.0. So
        identity adds almost nothing on class 1 (0.8986 -> 0.8955, a rounding
        error) while costing a great deal on class 2, where the true answer has
        no spectrum and scores 0 but wrong candidates that happen to be in the
        library score above it.

        `self.analog_weight` is retained so score_components can be re-tested if
        a future channel changes the balance, but it no longer affects ranking.
        """
        cand, ident, analog = self.score_components(spectra, adduct, ppm, exclude)
        if len(cand) == 0:
            return cand, np.empty(0, dtype=np.float32)

        if self.fpnet is None:
            total = analog
        else:
            # Swept on 290 natural-product structures, read on the hard bucket
            # (best analog Tanimoto < 0.4) because that is the regime the real
            # test set is mostly in:
            #   analog alone          0.6427 all / 0.4246 hard
            #   fpnet alone           0.6946      / 0.6307
            #   fpnet + 0.25*analog   0.7424      / 0.6567   <- chosen
            #   fpnet + 1.00*analog   0.7105      / 0.5489
            # A small analog weight adds real evidence; an equal one drowns the
            # model exactly where the model is the only thing that works.
            logits = predict_logits(
                self.fpnet, spectra, adduct, self.adducts,
                instrument_id_=0,   # the test set is 100% timsTOF
                device=self.device,
                max_peaks=self.max_peaks,
                collision_energies=collision_energies,
            )
            fp_score = score_candidates(
                logits, candidate_bits(self.pool.smiles[cand], self.bit_sel)
            )
            total = _z(fp_score) + FPNET_ANALOG_WEIGHT * _z(analog)

        n = min(k, len(cand))
        top = np.argpartition(-total, n - 1)[:n]
        top = top[np.argsort(-total[top])]
        out, scores = cand[top], total[top]

        if len(out) < k:
            out, scores = self._pad(out, scores, spectra[0][0], adduct, ppm, k)
        return out, scores

    def _pad(self, out, scores, mz, adduct, ppm, k):
        """Fill unused slots with the next mass-nearest structures.

        A tight mass window can return fewer than k candidates — sometimes as
        few as 3. The metric charges nothing for a wrong guess beyond the rank
        it occupies, so leaving 22 slots empty forfeits 22 free chances. Widen
        the tolerance and append by mass proximity.

        These are appended AFTER the ranked candidates, so they can never
        displace a scored one.
        """
        target = self.pool.neutral_mass(mz, adduct, self.offset_ppm)
        if target is None:
            return out, scores
        wide = self.pool.query(mz, adduct, ppm=ppm * 20, offset_ppm=self.offset_ppm)
        if len(wide) == 0:
            return out, scores

        # np.isin(..., invert=True) drops the ones already ranked.
        extra = wide[np.isin(wide, out, invert=True)]
        if len(extra) == 0:
            return out, scores

        extra = extra[np.argsort(np.abs(self.pool.mass[extra] - target))]
        extra = extra[: k - len(out)]
        return (
            np.concatenate([out, extra]),
            np.concatenate([scores, np.zeros(len(extra), dtype=np.float32)]),
        )

## Step 7 — build the index

~7 minutes and ~10 GB on Kaggle's 4 vCPUs, against a 9-hour cap. Written to
`/tmp`, not `/kaggle/working`, because it exceeds 2 GB and `working/` is
collected as the notebook's output.

In [ ]:
import time

import numpy as np
import pandas as pd

def find_one(name: str) -> Path:
    """Locate a required input by filename, or fail with a usable message.

    `next(INPUT.rglob(name))` raises a bare StopIteration that says nothing
    about what went wrong. v6 died that way: the kernel had mounted the
    PREVIOUS version of the deps dataset, which predated fpnet.pt. The dataset
    reported `ready` before the push, so `ready` is not sufficient — the
    version takes longer to reach the kernel runtime.

    Listing what IS present turns a one-word exception into a diagnosis.
    """
    hits = sorted(INPUT.rglob(name))
    if not hits:
        have = sorted(p.name for p in INPUT.rglob("*") if p.is_file())[:40]
        raise FileNotFoundError(
            f"{name!r} not found under /kaggle/input. If this is a dataset file "
            f"just uploaded, the kernel may have mounted an older version — "
            f"re-push after the new version propagates. Present: {have}"
        )
    return hits[0]


# Find inputs by filename rather than assuming the mount path.
TRAIN = find_one("train.parquet")
COMP = TRAIN.parent
POOL = find_one("pool.parquet")
FPNET = find_one("fpnet.pt")
print("competition data: ", COMP)
print("candidate pool:   ", POOL)
print("fingerprint model:", FPNET)

INDEX = Path("/tmp/index")
OUT = Path("/kaggle/working/submission.csv")
K = 25

t0 = time.time()
build(TRAIN, INDEX, TEST_MZ_RANGE, set(TEST_ADDUCTS))
print(f"index built in {time.time() - t0:.0f}s")

## Step 8 — rank every test molecule

A molecule may have been measured several times at different collision
energies. Scoring is per molecule, so `rank` fuses them, keeping each
structure's best analog score across the measurements.

In [ ]:
import torch

lib = Library(INDEX)
pool = CandidatePool(POOL)

# torch ships in the Kaggle base image, so no wheel is needed for it.
DEV = "cuda" if torch.cuda.is_available() else "cpu"
fpnet, fpnet_ck = load(FPNET, device=DEV)
print(f"fpnet: {fpnet_ck['cfg']['n_bits']} bits, "
      f"max_peaks {fpnet_ck.get('max_peaks')}, "
      f"val loss {fpnet_ck.get('val_loss', float('nan')):.4f}, "
      f"val F1 {fpnet_ck.get('val_f1', float('nan')):.3f}, running on {DEV}")

# Kaggle can mount a STALE dataset version — v6 failed exactly that way, with
# the listing showing the new file while the kernel had the old one. Assert the
# model is the one we meant to ship, so a stale mount fails in seconds instead
# of quietly scoring an 18-minute run with the wrong weights.
assert fpnet_ck.get("max_peaks") == 256, (
    f"expected the 256-peak model, got max_peaks={fpnet_ck.get('max_peaks')}. "
    f"Kaggle has probably mounted an older version of casmi26-deps; re-push."
)
ranker = CombinedRanker(lib, pool, fpnet=fpnet, fpnet_ck=fpnet_ck, device=DEV)

test = pd.read_parquet(COMP / "test.parquet")
print(f"test:    {len(test):,} spectra / {test.molecule_id.nunique():,} molecules")
print(f"library: {len(lib.meta):,} spectra / {lib.n_structures:,} structures")
print(f"pool:    {len(pool):,} structures")


def peaks_of(row) -> np.ndarray:
    """One test row's two parallel arrays as the (N, 2) shape ms_entropy wants."""
    return np.stack(
        [
            np.asarray(row.ms2_mzs, dtype=np.float32),
            np.asarray(row.ms2_normalized_intensities, dtype=np.float32),
        ],
        axis=1,
    )

In [ ]:
t0 = time.time()
rows, no_cand, short = [], 0, 0

# sort=True keeps the output order deterministic between runs.
for mid, grp in test.groupby("molecule_id", sort=True):
    spectra = [(float(r.precursor_mz), peaks_of(r)) for r in grp.itertuples()]
    adduct = grp.adduct.iloc[0]

    # Real collision energies, averaged over a stepped acquisition exactly as
    # training did. Passing 0 instead cost the previous submission: the model
    # saw a median of 32 eV in training and only 14.5% zeros.
    ces = [
        float(np.mean(r.collision_energy_ev))
        if r.collision_energy_ev is not None and len(r.collision_energy_ev)
        else 32.0
        for r in grp.itertuples()
    ]

    cand, scores = ranker.rank(spectra, adduct, k=K, collision_energies=ces)

    if len(cand) == 0:
        # No candidate of the right mass — an unrecognised adduct, or a mass
        # outside the pool. Never leave the row empty: a wrong guess costs
        # nothing beyond the rank it occupies, and an empty field is rejected.
        no_cand += 1
        smiles = list(pool.smiles[:K])
    else:
        smiles = list(pool.smiles[cand])

    if len(smiles) < K:
        short += 1
    rows.append({"molecule_id": mid, "smiles": ";".join(smiles)})

sub = pd.DataFrame(rows)
print(f"ranked {len(sub):,} molecules in {time.time() - t0:.0f}s")
print(f"  no mass-matched candidates: {no_cand} ({no_cand / len(sub):.1%})")
print(f"  fewer than {K} candidates:  {short} ({short / len(sub):.1%})")

## Step 9 — validate the submission contract

The host rejects a file that repeats a `molecule_id`, contains nulls, or gives
more than 25 guesses. A rejection costs a submission slot and a day, so assert
every rule before writing.

In [ ]:
assert not sub.molecule_id.duplicated().any(), "duplicate molecule_id"
assert sub.smiles.notna().all() and sub.molecule_id.notna().all(), "null value"

counts = sub.smiles.str.count(";") + 1   # guesses = separators + 1
assert counts.max() <= K, f"{counts.max()} guesses for one molecule, max {K}"
assert counts.min() >= 1, "a molecule has no guesses"
assert set(sub.molecule_id) == set(test.molecule_id), "molecule_id set mismatch"

# index=False stops pandas writing row numbers as a third, unnamed column.
sub.to_csv(OUT, index=False)

print(f"wrote {OUT}")
print(f"  {len(sub):,} rows, {counts.min()}-{counts.max()} guesses each")
print(sub.head(3).to_string())